## carregando imagem

In [1]:
import cv2 as cv
def mostrar_imagem(img):

    cv.imshow("amostra de imagem",img)
    cv.waitKey(0)
    cv.destroyAllWindows()

In [ ]:
import cv2
import numpy as np
import os

class BandejaProcessor:
    def __init__(self, pasta_saida="pecas", width=800, height=600):
        self.pasta_saida = pasta_saida
        self.width = width
        self.height = height
        os.makedirs(self.pasta_saida, exist_ok=True)
        
        # Pontos de destino para a bandeja perfeitamente plana
        self.pts_destino = np.float32([
            [0, 0], 
            [self.width, 0], 
            [0, self.height], 
            [self.width, self.height]
        ])

    def _padronizar(self, caminho_imagem):
        # Lê a imagem e redimensiona para o tamanho padrão (800x600)[cite: 1]
        img = cv2.imread(caminho_imagem)
        if img is None:
            raise ValueError("Erro ao ler a imagem.")
        return cv2.resize(img, (self.width, self.height))

    def _ordenar_pontos(self, pontos):
        rect = np.zeros((4, 2), dtype="float32")
        s = pontos.sum(axis=1)
        rect[0] = pontos[np.argmin(s)]
        rect[2] = pontos[np.argmax(s)]
        diff = np.diff(pontos, axis=1)
        rect[1] = pontos[np.argmin(diff)]
        rect[3] = pontos[np.argmax(diff)]
        return rect

    def _detectar_cantos(self, img_resized):
        """
        AJUSTE 1: Isola o papel branco da mesa de madeira usando Otsu Threshold
        em vez de tentar achar bordas com Canny.
        """
        

        img_gray = cv2.cvtColor(img_resized, cv2.COLOR_BGR2GRAY)
        
        # Blur pesado para suavizar os veios da madeira e os objetos no papel
        img_blur = cv2.GaussianBlur(img_gray, (11, 11), 0)
        
        # Usa Limiarização de Otsu para separar perfeitamente o branco do escuro
        _, mask_papel = cv2.threshold(img_blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        
        # Encontra o maior contorno (que será a folha de papel)
        contornos, _ = cv2.findContours(mask_papel, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        contorno_maior = max(contornos, key=cv2.contourArea)
        
        perimetro = cv2.arcLength(contorno_maior, True)
        aproximacao = cv2.approxPolyDP(contorno_maior, 0.02 * perimetro, True)
        
        if len(aproximacao) == 4:
            return self._ordenar_pontos(aproximacao.reshape(4, 2))
            
        # Fallback se algo der muito errado
        print("Aviso: Papel não detectado. Usando pontos de fallback.")
        return np.float32([[100, 150], [700, 150], [50, 500], [750, 500]])

    def _corrigir_perspectiva(self, img_resized, pts_origem):
        # Realiza o warp perspective (envelopamento) com os 4 pontos detectados[cite: 1]
        matriz = cv2.getPerspectiveTransform(pts_origem, self.pts_destino)
        return cv2.warpPerspective(img_resized, matriz, (self.width, self.height))

    def _pre_processar(self, img_warped):
        """
        AJUSTE 2: Inversão de cores para atender ao PDF.
        """
        # Converte a imagem envelopada para Tons de Cinza[cite: 1]
        img_gray = cv2.cvtColor(img_warped, cv2.COLOR_BGR2GRAY)
        
        # Aplica um desfoque suave para remover ruídos na superfície das peças[cite: 1]
        img_blur = cv2.GaussianBlur(img_gray, (5, 5), 0)
        
        # Aplica limiar para criar máscara binária (fundo preto e objetos brancos)[cite: 1]
        # Como o fundo é branco e as peças são escuras, usamos THRESH_BINARY_INV com Otsu.
        _, mask = cv2.threshold(img_blur, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
        return img_gray, mask

    def _aplicar_morfologia(self, mask):
        # Operações morfológicas para limpar a máscara[cite: 1]
        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
        
        # Abertura (Opening) remove sujeiras externas do papel (ruídos isolados)[cite: 1]
        mask_limpa = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
        
        # Fechamento (Closing) preenche buracos e reflexos de luz dentro das peças[cite: 1]
        mask_limpa = cv2.morphologyEx(mask_limpa, cv2.MORPH_CLOSE, kernel)
        return mask_limpa

    def _extrair_e_salvar(self, caminho_imagem, img_warped, img_gray, mask):
        # Encontra os contornos na máscara morfológica limpa[cite: 1]
        contornos, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        nome_base = os.path.splitext(os.path.basename(caminho_imagem))[0]
        
        img_visualizacao = img_gray.copy()

        for i, contorno in enumerate(contornos):
            # Filtro rigoroso: Ignora ruídos menores que 300 pixels
            if cv2.contourArea(contorno) < 300:
                continue
                
            # Desenha retângulo delimitador (Bounding Box) na imagem cinza[cite: 1]
            x, y, w, h = cv2.boundingRect(contorno)
            cv2.rectangle(img_visualizacao, (x, y), (x+w, y+h), (255, 255, 255), 2)
            
            # Recorta a região de cada peça e salva o arquivo[cite: 1]
            peca_recorte = img_warped[y:y+h, x:x+w]
            nome_arquivo = f"{nome_base}_peca_{i+1}.jpg"
            cv2.imwrite(os.path.join(self.pasta_saida, nome_arquivo), peca_recorte)

    def processar_imagem(self, caminho_imagem):
        """Orquestrador do pipeline."""
        img_resized = self._padronizar(caminho_imagem)
        pts_origem = self._detectar_cantos(img_resized)
        img_warped = self._corrigir_perspectiva(img_resized, pts_origem)
        img_gray, mask = self._pre_processar(img_warped)
        mask_limpa = self._aplicar_morfologia(mask)
        self._extrair_e_salvar(caminho_imagem, img_warped, img_gray, mask_limpa)
        return True

In [7]:
processor = BandejaProcessor(pasta_saida='output')
processor.processar_imagem('data/raw/Exer_1.jpeg')


Aviso: Papel não detectado. Usando pontos de fallback.


QObject::moveToThread: Current thread (0x27747790) is not the object's thread (0x281f3aa0).
Cannot move to target thread (0x27747790)

QObject::moveToThread: Current thread (0x27747790) is not the object's thread (0x281f3aa0).
Cannot move to target thread (0x27747790)

QObject::moveToThread: Current thread (0x27747790) is not the object's thread (0x281f3aa0).
Cannot move to target thread (0x27747790)

QObject::moveToThread: Current thread (0x27747790) is not the object's thread (0x281f3aa0).
Cannot move to target thread (0x27747790)

QObject::moveToThread: Current thread (0x27747790) is not the object's thread (0x281f3aa0).
Cannot move to target thread (0x27747790)

QObject::moveToThread: Current thread (0x27747790) is not the object's thread (0x281f3aa0).
Cannot move to target thread (0x27747790)

QObject::moveToThread: Current thread (0x27747790) is not the object's thread (0x281f3aa0).
Cannot move to target thread (0x27747790)

QObject::moveToThread: Current thread (0x27747790) is n

True